[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S05/2_Practica/en_clase/S05_04_estadistica_descriptiva_ic_en_clase.ipynb)

# S05_04 · Estadística con Python: describir y dar un margen de error (en clase)

<h2 align="center">Resumir 1096 días de ocupación y decir cuánto fiarse de una cifra</h2>

| | |
|---|---|
| ⏱ **Cuándo** | Jueves · unos 40 min |
| 👥 **Cómo** | En parejas: una persona escribe, la otra lee y comprueba (cambiad de papel al empezar la parte B) |
| 🧰 **Usáis** | 🌐 Google Colab (este cuaderno) · no hay que instalar nada |
| 🏁 **Al terminar** | Ejercicio 1 (resumen de la serie) y ejercicio 5 (intervalo de confianza) con todo ✅ en el paso 5 |

**Este cuaderno tiene dos partes:**

- **1 · Guía paso a paso**: solo lo que hay que hacer. Va en dos bloques: **A · Describir** y **B · Intervalo de confianza**.
- **2 · Explicaciones**: el porqué de cada cosa, para cuando lo necesitéis (está al final, también en dos bloques: A y B).
- Si queréis, seguid solo la parte 1.

**Solo trae los dos ejercicios de clase**: el **1 trata de describir** una serie de datos (parte A) y el **5 del intervalo de confianza** (parte B). Los demás (2, 3, 4 y el mini-reto) están en el cuaderno completo [`S05_04_estadistica_descriptiva_ic`](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S05/2_Practica/S05_04_estadistica_descriptiva_ic.ipynb), para casa.

## Cómo se usa este cuaderno (leedlo una vez)

**Un cuaderno es una mezcla de textos y celdas de código.** Las celdas de código tienen a la izquierda un botón **▶**.

1. **Ejecutar una celda**
   - Pulsad el botón **▶** de la izquierda, o haced clic dentro de la celda y pulsad **Mayús + Intro**.
   - Debajo de la celda aparece **lo que ha salido**.
2. **De arriba abajo, en orden**
   - Cada celda usa lo que han hecho las anteriores.
   - Si os saltáis una, saldrá un error rojo que dice `NameError`. No pasa nada: ejecutad las que faltan y repetid.
3. **Las celdas con `...`**
   - Son los ejercicios. Los **tres puntos `...` se sustituyen** por vuestro código.
   - No borréis el resto de la celda.
4. **Cada paso termina en ✅**
   - Es lo que **tiene que salir**.
   - Los decimales lejanos pueden cambiar un poco en vuestro ordenador; lo demás tiene que ser **igual**.
   - Si no sale, no sigáis: mirad la parte 2, apartado «Si os atascáis».
5. **Si Colab se reinicia o cerráis la pestaña**
   - Se pierde lo ejecutado, no lo escrito.
   - Volved a ejecutar las celdas **desde el paso 0 hasta donde estabais**.

# 1 · Guía paso a paso

**Qué vais a hacer**

- **Parte A · Describir (ejercicio 1):** resumir la ocupación diaria de **1096 días** con cuatro números: la media, la mediana, la desviación y el rango intercuartílico.
- **Parte B · Intervalo de confianza (ejercicio 5):** calcular la **tasa de cancelación** de 6000 reservas y darle un **margen de error**, para saber cuánto fiarse de esa cifra.
- Son las dos preguntas de la dirección del alojamiento: «¿cuál es la ocupación típica?» y «¿con cuánta seguridad podemos dar una cifra?».

## Antes de empezar · Paso 0 · Preparar el cuaderno (3 min)

| # | Qué hacéis |
|---|---|
| **0.1** | Ejecutad la celda de abajo (la de «Preparación»). Tarda unos segundos: busca los ficheros de datos y, si no están, los descarga. |
| **0.2** | Ejecutad la celda siguiente (la de las librerías). |

✅ Con la primera sale `Preparación lista ✅` y con la segunda `Librerías listas ✅`.

*No hace falta entender estas dos celdas: dejan preparado lo que usaremos después.*

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

RESULTADOS = {}

def comprobar(nombre, condicion, ayuda):
    """Escribe ✅ si la condición es cierta y ❌ con una pista si no. No hace falta entender cómo funciona."""
    try:
        ok = bool(condicion())
    except Exception:
        ok = False
    RESULTADOS[nombre] = ok
    print(("✅ " + nombre) if ok else ("❌ " + nombre + " → " + ayuda))

dato("ocupacion_diaria.csv")
dato("reservas_turisdata.csv")
print("Preparación lista ✅")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

print("Librerías listas ✅")

# Parte A · Describir (ejercicio 1) · unos 15 min

**Qué vais a hacer**

- Tenéis la **ocupación de cada día** durante 3 años: 1096 números. Nadie puede leerlos todos.
- **Describir** es resumirlos con pocos números que digan cómo es la serie.
- Vais a calcular cuatro: la **media** y la **mediana** (el centro), la **desviación** y el **rango intercuartílico** (cuánto se separan los datos).
- Primero lo veréis con 5 números inventados (paso 2) y después lo haréis con los datos reales: es el **ejercicio 1**.

## Paso 1 · Cargar la serie y mirarla (5 min)

**Los datos**

- Una **fila por día**, con cinco columnas: `fecha`, `ocupacion_pct`, `temperatura_c`, `festivo` y `precio_medio`.
- Hoy solo usamos la **ocupación** (`ocupacion_pct`): la llamaremos `serie`.

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **1.1** | Ejecutad la celda 1.1 (carga el fichero en una tabla llamada `ocupacion_df`). | `(1096, 5)` |
| **1.2** | Ejecutad la celda 1.2 (guarda la ocupación en `serie` y enseña los 5 primeros días). | `[89.0, 84.7, 71.6, 85.6, 78.7]` |
| **1.3** | Ejecutad la celda 1.3 (`describe()`: un resumen automático). | Una lista con `count 1096.00`, `mean 68.49`, `std 9.61`, `50% 67.70`… |

- `(1096, 5)` quiere decir: **1096 filas** (días) y **5 columnas**.
- En 1.3 ya aparecen los números del ejercicio: la **media** (`mean`), la **mediana** (`50%`) y la **desviación** (`std`).

In [ ]:
# 1.1 · Cargar la tabla
ocupacion_df = pd.read_csv(dato("ocupacion_diaria.csv"), parse_dates=["fecha"])
print(ocupacion_df.shape)

In [ ]:
# 1.2 · La serie de ocupación y los 5 primeros días
serie = ocupacion_df["ocupacion_pct"]
print(serie.head().tolist())

In [ ]:
# 1.3 · Resumen automático
print(serie.describe().round(2))

## Paso 2 · Ejercicio 1 · Resumen de la ocupación (10 min) 🏫

**Objetivo**

- Guardar en un diccionario llamado `resumen` cuatro números de la serie, **redondeados a 2 decimales**.
- Un **diccionario** es una lista con etiquetas: `"media"` → 68.49, `"mediana"` → 67.7…

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **2.1** | **A mano, con 5 números** (60, 70, 70, 80 y 100). Ejecutad la celda 2.1. | `Media: 76.0`, `Mediana: 70.0`, `Desviación: 15.17` y `IQR: 10.0` |
| **2.2** | **Vuestro turno.** En la celda 2.2, sustituid cada `...` por la misma cuenta pero con `serie`, dentro de `round(…, 2)`. | No sale nada |
| **2.3** | Ejecutad la celda 2.3 (la comprobación). | Cuatro líneas con ✅ |
| **2.4** | Ejecutad la celda 2.4 (muestra el `resumen` y dibuja la serie con la media y la mediana). | `media: 68.49`, `mediana: 67.7`, `desviacion: 9.61`, `iqr: 13.6` y un gráfico de barras con dos líneas |

- **Pistas para 2.2:**
  - `"media"`: `round(serie.mean(), 2)`
  - `"mediana"`: `round(serie.median(), 2)`
  - `"desviacion"`: `round(serie.std(), 2)`
  - `"iqr"`: `round(serie.quantile(0.75) - serie.quantile(0.25), 2)`
- Los números de 2.1 enseñan una cosa: la **media (76) es mayor que la mediana (70)** porque el `100` tira de ella hacia arriba.
- Si sale ❌, leed la pista que aparece tras la flecha `→` y corregid.

In [ ]:
# 2.1 · A mano, con cinco números
pocos = pd.Series([60, 70, 70, 80, 100])
print("Media:", pocos.mean())                                   # la suma entre cuántos son
print("Mediana:", pocos.median())                               # el valor del medio, con los datos ordenados
print("Desviación:", round(pocos.std(), 2))                     # cuánto se separan de la media
print("IQR:", pocos.quantile(0.75) - pocos.quantile(0.25))      # percentil 75 menos percentil 25

In [ ]:
# 2.2 · TU TURNO: lo mismo que en la celda 2.1, pero con serie y redondeado a 2 decimales
resumen = {
    "media": ...,         # round(serie.mean(), 2)
    "mediana": ...,       # round(serie.median(), 2)
    "desviacion": ...,    # round(serie.std(), 2)
    "iqr": ...,           # round(serie.quantile(0.75) - serie.quantile(0.25), 2)
}

In [ ]:
# 2.3 · Comprobación del ejercicio 1 (no hay que cambiar nada)
comprobar("resumen: media", lambda: resumen["media"] == 68.49, "la media debe salir 68.49: round(serie.mean(), 2). ¿Has cambiado los puntos ... por tu código?")
comprobar("resumen: mediana", lambda: resumen["mediana"] == 67.7, "la mediana debe salir 67.7: round(serie.median(), 2)")
comprobar("resumen: desviación", lambda: resumen["desviacion"] == 9.61, "la desviación debe salir 9.61: round(serie.std(), 2)")
comprobar("resumen: IQR", lambda: resumen["iqr"] == 13.6, "el IQR debe salir 13.6: percentil 75 menos percentil 25, es decir serie.quantile(0.75) - serie.quantile(0.25)")

In [ ]:
# 2.4 · Mirar el resultado
for nombre, valor in resumen.items():
    print(f"{nombre}: {valor}")
plt.figure(figsize=(7, 3.5))
serie.hist(bins=20, color="lightsteelblue", edgecolor="white")
plt.axvline(resumen["media"], color="crimson", label=f"Media {resumen['media']}")
plt.axvline(resumen["mediana"], color="darkgreen", linestyle="--", label=f"Mediana {resumen['mediana']}")
plt.xlabel("Ocupación diaria (%)")
plt.ylabel("Número de días")
plt.title("Cómo se reparte la ocupación de los 1096 días")
plt.legend()
plt.tight_layout()
plt.show()

✅ Tiene que salir: `✅ resumen: media`, `✅ resumen: mediana`, `✅ resumen: desviación` y `✅ resumen: IQR`. Si falta alguno, no sigáis.

# Parte B · Intervalo de confianza (ejercicio 5) · unos 20 min

*Cambiad de papel: quien escribía, ahora lee y comprueba.*

**Qué es un intervalo de confianza y qué vais a hacer**

- Con los datos solo tenemos una **estimación**. Si el 45,9 % de las reservas se cancelan, ¿cuánto podemos fiarnos de ese número?
- Un **intervalo de confianza (IC)** acompaña la cifra con un **margen de error**: un rango de valores razonables, por ejemplo «entre 44,6 % y 47,2 %».
- Primero **veréis cómo se calcula** con un ejemplo pequeño (paso 3).
- Después, el **ejercicio 5**: el IC de la tasa de cancelación de las 6000 reservas (paso 4).

## Paso 3 · La idea del intervalo (7 min)

**Objetivo**

- Mirar los datos de las reservas.
- Ver el cálculo del intervalo con un ejemplo de **100 reservas**.

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **3.1** | Ejecutad la celda 3.1 (carga las reservas; la columna `cancelada` vale `1` si se canceló y `0` si no). | `(6000, 18)` y `[1, 0, 1, 0, 0]` |
| **3.2** | **A mano, con 100 reservas** de las que 40 se cancelan. Ejecutad la celda 3.2. | `p = 0.4 · error = 0.049` y `IC 95 %: (0.304, 0.496)` |
| **3.3** | Ejecutad la celda 3.3: el mismo tipo de intervalo, pero para la **ocupación media**. | `Media 68.49 · IC 95 %: (67.92, 69.06)` |

- **La fórmula del IC de una proporción:** `p ± 1.96 × error`, con `error = √(p (1 − p) / n)`.
- En 3.2 la tasa es `0.4`, pero con solo 100 reservas el intervalo es **ancho**: entre `0.304` y `0.496`.
- Cuantas **más reservas**, más estrecho el intervalo. Con 6000 será mucho más fino.

In [ ]:
# 3.1 · Cargar las reservas
reservas = pd.read_csv(dato("reservas_turisdata.csv"))
print(reservas.shape)
print(reservas["cancelada"].head().tolist())

In [ ]:
# 3.2 · A mano: 100 reservas, 40 canceladas
p = 40 / 100                                    # la proporción de canceladas
n = 100                                         # cuántas reservas hay
error = np.sqrt(p * (1 - p) / n)                # el error estándar
print(f"p = {p} · error = {error:.3f}")
print(f"IC 95 %: ({p - 1.96 * error:.3f}, {p + 1.96 * error:.3f})")

In [ ]:
# 3.3 · Ejemplo ya hecho: el IC de la ocupación media (con SciPy)
n_dias = len(serie)
ic_media = stats.t.interval(0.95, n_dias - 1, loc=serie.mean(), scale=stats.sem(serie))
print(f"Media {serie.mean():.2f} · IC 95 %: ({ic_media[0]:.2f}, {ic_media[1]:.2f})")

## Paso 4 · Ejercicio 5 · IC de la tasa de cancelación (10 min) 🏫

**Objetivo**

- Calcular la proporción de reservas canceladas y su intervalo de confianza del 95 %.
- Es la misma cuenta de la celda 3.2, pero con las 6000 reservas.

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **4.1** | **Vuestro turno.** En la celda 4.1, sustituid los `...` por las mismas cuentas de la celda 3.2, con los datos de `reservas`. | No sale nada |
| **4.2** | Ejecutad la celda 4.2 (la comprobación). | Cuatro líneas con ✅ |
| **4.3** | Ejecutad la celda 4.3 (muestra el resultado en porcentajes). | `Tasa de cancelación: 45.9 % (IC 95 %: 44.6 % a 47.2 %)` |

- **Pistas para 4.1:**
  - `p_hat = reservas["cancelada"].mean()` (la media de unos y ceros es la proporción de unos)
  - `n_res = len(reservas)`
  - `error = np.sqrt(p_hat * (1 - p_hat) / n_res)`
  - `ic_tasa = (round(p_hat - 1.96 * error, 4), round(p_hat + 1.96 * error, 4))`
- Si sale ❌, leed la pista que aparece tras la flecha `→` y corregid.
- El resultado de 4.3 dice: **la tasa de cancelación real está entre el 44,6 % y el 47,2 %** con un 95 % de confianza.

In [ ]:
# 4.1 · TU TURNO: las mismas cuentas de la celda 3.2, pero con las 6000 reservas
p_hat = ...                    # la proporción de canceladas: reservas["cancelada"].mean()
n_res = ...                    # cuántas reservas hay: len(reservas)
error = ...                    # np.sqrt(p_hat * (1 - p_hat) / n_res)
ic_tasa = (..., ...)           # (p_hat - 1.96 * error, p_hat + 1.96 * error), cada uno con round(..., 4)

In [ ]:
# 4.2 · Comprobación del ejercicio 5 (no hay que cambiar nada)
comprobar("IC: p_hat", lambda: abs(p_hat - 0.459) < 1e-9, "p_hat debe ser 0.459: reservas[\"cancelada\"].mean(). ¿Has cambiado los puntos ... por tu código?")
comprobar("IC: n_res", lambda: n_res == 6000, "n_res debe ser 6000: len(reservas)")
comprobar("IC: error", lambda: abs(error - 0.00643) < 1e-4, "el error debe salir 0.0064: np.sqrt(p_hat * (1 - p_hat) / n_res)")
comprobar("IC: intervalo", lambda: ic_tasa == (0.4464, 0.4716), "el intervalo debe ser (0.4464, 0.4716): (p_hat - 1.96 * error, p_hat + 1.96 * error), cada uno con round(..., 4)")

In [ ]:
# 4.3 · Mirar el resultado
print(f"Tasa de cancelación: {100 * p_hat:.1f} % (IC 95 %: {100 * ic_tasa[0]:.1f} % a {100 * ic_tasa[1]:.1f} %)")

## Paso 5 · Cerrar los dos ejercicios (2 min)

| # | Qué hacéis | ✅ Tiene que salir |
|---|---|---|
| **5.1** | Ejecutad la celda 5.1 (cuenta las comprobaciones ✅ de cada ejercicio por separado). | `Parte A · Describir (ejercicio 1): 4 de 4 ✅`, `Parte B · Intervalo de confianza (ejercicio 5): 4 de 4 ✅` y `🎉 Los ejercicios 1 y 5 están hechos` |
| **5.2** | Si os sobra tiempo, abrid el cuaderno completo [`S05_04_estadistica_descriptiva_ic`](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S05/2_Practica/S05_04_estadistica_descriptiva_ic.ipynb) y probad los ejercicios 2 (ocupación por meses) y 3 (probabilidades con la binomial). | — |

In [ ]:
# 5.1 · Resumen por ejercicio
N1 = ["resumen: media", "resumen: mediana", "resumen: desviación", "resumen: IQR"]
N5 = ["IC: p_hat", "IC: n_res", "IC: error", "IC: intervalo"]
bien1 = sum(RESULTADOS.get(n, False) for n in N1)
bien5 = sum(RESULTADOS.get(n, False) for n in N5)
print(f"Parte A · Describir (ejercicio 1): {bien1} de 4 ✅")
print(f"Parte B · Intervalo de confianza (ejercicio 5): {bien5} de 4 ✅")
if bien1 == 4 and bien5 == 4:
    print("🎉 Los ejercicios 1 y 5 están hechos")
else:
    print("Faltan comprobaciones: mirad las líneas ❌ de las celdas 2.3 (ejercicio 1) y 4.2 (ejercicio 5)")

## ✅ Terminado cuando

**Parte A · Describir (ejercicio 1)**

- [ ] La celda 2.3 muestra las cuatro líneas ✅.
- [ ] Cada persona sabe explicar con sus palabras **qué diferencia hay entre la media y la mediana**.

**Parte B · Intervalo de confianza (ejercicio 5)**

- [ ] La celda 4.2 muestra las cuatro líneas ✅.
- [ ] Cada persona sabe explicar con sus palabras **qué quiere decir «entre 44,6 % y 47,2 %»**.

**Los dos**

- [ ] La celda 5.1 dice `🎉 Los ejercicios 1 y 5 están hechos`.

---

# 2 · Explicaciones

*Para cuando una cosa no sale o queréis entender por qué se hace así. No hace falta leerlo para resolver los ejercicios. Está en dos bloques, igual que la guía: **A · Describir** y **B · Intervalo de confianza**.*

# Explicaciones de la parte A · Describir (ejercicio 1)

## La idea: resumir muchos números con pocos

- La **estadística descriptiva** resume una serie de datos con unos pocos números.
- Se mira en dos cosas:
  - **El centro:** ¿alrededor de qué valor están los datos? → **media** y **mediana**.
  - **La dispersión:** ¿cuánto se separan unos de otros? → **desviación típica** y **rango intercuartílico**.
- Con 1096 días, la pregunta «¿cuál es la ocupación típica y cuánto varía?» se contesta con estos cuatro números.

## Los cuatro números, con 5 datos de ejemplo

Los datos, ya ordenados: **60, 70, 70, 80, 100**.

| Número | Cómo se calcula | Con el ejemplo | Código |
|---|---|---|---|
| **Media** | Suma de todos entre cuántos son | (60+70+70+80+100) ÷ 5 = **76** | `serie.mean()` |
| **Mediana** | El valor del medio, con los datos ordenados | El tercero de cinco: **70** | `serie.median()` |
| **Desviación típica** | Cuánto se separan los datos de la media, de media | **15.17** | `serie.std()` |
| **IQR** | Percentil 75 menos percentil 25 | 80 − 70 = **10** | `serie.quantile(0.75) - serie.quantile(0.25)` |

- **Percentil 25 y 75 (cuartiles):**
  - El **percentil 25** es el valor que deja a su izquierda el 25 % de los datos.
  - El **percentil 75** deja el 75 %. Entre los dos queda **la mitad central** de los datos.
  - El IQR es lo ancha que es esa mitad central.
- **¿Por qué la media (76) es mayor que la mediana (70)?**
  - Porque el `100` es muy grande y **tira de la media** hacia arriba.
  - La mediana no se mueve: solo mira el valor del medio.
- Con los datos reales la media y la mediana son parecidas (68.49 y 67.7): la serie está bastante **equilibrada**.

## Cómo se lee el diccionario `resumen`

- Un **diccionario** es una colección de pares **etiqueta → valor**, entre llaves `{}`.
- Para pedir un valor se usa su etiqueta entre corchetes: `resumen["media"]` → `68.49`.
- `round(número, 2)` redondea a **2 decimales**.
- El histograma de 2.4 es un **gráfico de barras**: cuántos días tienen cada nivel de ocupación.
  - La línea roja es la media y la verde discontinua la mediana.
  - Casi coinciden: los datos se reparten de forma simétrica.

## Las palabras de la parte A

| Palabra | Qué significa |
|---|---|
| **Serie** | Una columna de datos, uno por día (`serie = ocupacion_df["ocupacion_pct"]`) |
| **Media** | La suma de los datos dividida entre cuántos son |
| **Mediana** | El valor del medio cuando los datos están ordenados |
| **Desviación típica (`.std()`)** | Cuánto se separan los datos de la media, de media |
| **Percentil / cuartil** | El valor que deja a su izquierda un tanto por ciento de los datos |
| **IQR** | Rango intercuartílico: percentil 75 menos percentil 25 |
| **`describe()`** | Un resumen automático con los números más usados |
| **Diccionario (`{}`)** | Una colección de pares etiqueta → valor |
| **`round(x, 2)`** | Redondea `x` a 2 decimales |
| **Histograma** | Gráfico de barras que cuenta cuántos datos caen en cada tramo |
| **✅ / ❌** | Lo que escribe la función `comprobar`: ✅ está bien; ❌ hay algo que corregir (con una pista) |

## Por qué se hace así

- **¿Por qué dos números para el centro (media y mediana)?**
  - Si coinciden, la serie está equilibrada.
  - Si no, hay valores extremos (muy grandes o muy pequeños) tirando de la media.
- **¿Por qué dos números para la dispersión?**
  - La desviación es la más usada. El IQR **no se deja engañar** por los valores extremos.
- **¿Por qué `std()` de pandas da 9.61 y no otro valor?**
  - Pandas divide entre `n − 1` en lugar de `n` (se llama `ddof=1`). Para muchos datos casi no cambia, y es la forma habitual cuando se quiere inferir.
- **¿Por qué redondear a 2 decimales?**
  - Para que todos obtengáis **exactamente** el mismo número y la comprobación pueda compararlo.

# Explicaciones de la parte B · Intervalo de confianza (ejercicio 5)

## La idea: una cifra necesita su margen de error

- Las 6000 reservas son una **muestra**: no son todas las reservas posibles. Si hubiera otras 6000, la tasa de cancelación saldría un poco distinta.
- Por eso una cifra suelta («45,9 %») engaña: parece exacta y no lo es.
- Un **intervalo de confianza** da un **rango razonable** para la cifra «verdadera»: **entre 44,6 % y 47,2 %**.
- Se dice «del 95 %» porque el método está hecho para **acertar 95 de cada 100 veces**: si repitiéramos el estudio muchas veces, 95 de cada 100 intervalos construidos así contendrían la tasa real.
- **Cuidado con la frase:** no significa «hay un 95 % de probabilidad de que la tasa esté aquí». Significa que **el método acierta el 95 % de las veces**.

## La fórmula del IC de una proporción, leída en partes

La fórmula es `p ± 1.96 × √(p (1 − p) / n)`.

| Pieza | Qué es | En el ejercicio |
|---|---|---|
| **`p_hat`** (`p`) | La proporción que hemos medido | 0.459 (el 45,9 % de reservas se cancelan) |
| **`n_res`** (`n`) | Cuántos datos hay | 6000 |
| **`error`** | `√(p (1 − p) / n)`: cuánto puede bailar `p` por azar | 0.0064 |
| **`1.96`** | El número que da el 95 % de confianza | Fijo |
| **`±`** | Restar para el extremo inferior, sumar para el superior | 0.459 − 1.96 × 0.0064 = **0.4464** · 0.459 + 1.96 × 0.0064 = **0.4716** |

- Con **más datos** (`n` grande) el error es **menor** y el intervalo es **más estrecho**:

| Reservas (`n`) | Tasa (`p`) | Intervalo del 95 % |
|---|---|---|
| 100 (ejemplo 3.2, con p = 0.4) | 0.4 | (0.304, 0.496) → **ancho** |
| 6000 (ejercicio 5) | 0.459 | (0.4464, 0.4716) → **estrecho** |

## Cómo se lee el resultado

- `Tasa de cancelación: 45.9 % (IC 95 %: 44.6 % a 47.2 %)` se lee:
  - «Hemos medido un **45,9 %** de cancelaciones.»
  - «La tasa real está **probablemente entre el 44,6 % y el 47,2 %**.»
- Para el negocio sirve para planificar: si el alojamiento hace *overbooking*, puede contar con cancelaciones **entre el 44 % y el 48 %**, no con un número exacto.
- `ic_tasa` es una **tupla**: dos valores entre paréntesis, `(inferior, superior)`. `ic_tasa[0]` es el inferior y `ic_tasa[1]`, el superior.

## Las palabras de la parte B

| Palabra | Qué significa |
|---|---|
| **Muestra** | Los datos que tenemos, que son solo una parte de todos los posibles |
| **Estimación** | Una cifra calculada con la muestra (el 45,9 %) |
| **Proporción (`p_hat`)** | La parte del total que cumple algo, entre 0 y 1 (0.459 = 45,9 %) |
| **Error estándar (`error`)** | Cuánto puede variar la estimación por azar |
| **Intervalo de confianza (IC)** | Un rango razonable para el valor verdadero |
| **Nivel de confianza (95 %)** | Con qué frecuencia acierta el método |
| **Tupla** | Dos o más valores entre paréntesis: `(0.4464, 0.4716)` |
| **`np.sqrt`** | Raíz cuadrada |
| **`len(reservas)`** | Cuántas filas tiene la tabla |

## Por qué se hace así

- **¿Por qué `.mean()` da la proporción?**
  - La columna `cancelada` solo tiene unos y ceros. La media de unos y ceros es **la fracción de unos**: 0.459 = el 45,9 %.
- **¿Por qué `p (1 − p)`?**
  - Mide cuánto «dudan» los datos. Si casi todas las reservas se cancelaran (p cerca de 1) o casi ninguna (p cerca de 0), habría poca duda y el error sería pequeño.
  - Es máximo cuando `p = 0.5`.
- **¿Por qué dividir entre `n`?**
  - Porque con más datos la estimación es más fiable. El error baja con la raíz de `n`: para reducirlo a la mitad hacen falta **cuatro veces** más datos.
- **¿Por qué 1.96?**
  - Es el valor de la campana normal que deja fuera el 2,5 % por cada lado, es decir, el 95 % en el centro.

## 🚑 Si os atascáis

**Parte A · Describir (ejercicio 1)**

| Qué pasa | Por qué | Qué hacer |
|---|---|---|
| `NameError: name 'serie' is not defined` | No se ha ejecutado la celda 1.2 | Ejecutad las celdas desde el paso 0 hasta la 1.2, en orden |
| `❌ resumen: …` en una sola línea | Esa cuenta tiene un fallo | Mirad la pista tras la flecha `→` y la lista de pistas del paso 2 |
| `❌ resumen: IQR` | Falta el paréntesis o se han restado al revés | `round(serie.quantile(0.75) - serie.quantile(0.25), 2)`: primero el 75, después el 25 |
| `❌ resumen: …` con un valor casi igual | Falta el `round(…, 2)` | Cada valor del diccionario va dentro de `round( , 2)` |
| `TypeError` o `AttributeError` que nombran `'ellipsis'` | Se ha quedado algún `...` sin sustituir | Cambiad los `...` por vuestro código |
| `SyntaxError` al ejecutar 2.2 | Falta una coma o una llave | Cada línea del diccionario acaba en coma; la última llave `}` va sola al final |

**Parte B · Intervalo de confianza (ejercicio 5)**

| Qué pasa | Por qué | Qué hacer |
|---|---|---|
| `NameError: name 'reservas' is not defined` | No se ha ejecutado la celda 3.1 | Ejecutad la celda 3.1 |
| `KeyError: 'cancelada'` | Se ha escrito mal el nombre de la columna | `reservas["cancelada"]`, con comillas y todo en minúsculas |
| `❌ IC: p_hat` | Falta `.mean()` o se ha usado otra columna | `reservas["cancelada"].mean()` |
| `❌ IC: n_res` | No se ha contado el número de reservas | `len(reservas)` |
| `❌ IC: error` | Falta el paréntesis o la raíz | `np.sqrt(p_hat * (1 - p_hat) / n_res)`, con **todo** dentro de la raíz |
| `❌ IC: intervalo` | Falta el `round(…, 4)` o se ha invertido el signo | `(round(p_hat - 1.96 * error, 4), round(p_hat + 1.96 * error, 4))`: primero el menos, después el más |
| `NameError: name 'p_hat' is not defined` en 4.3 | La celda 4.1 no se ha ejecutado | Ejecutad 4.1 y después 4.2 y 4.3 |

**En cualquier parte**

| Qué pasa | Por qué | Qué hacer |
|---|---|---|
| `NameError: name 'pd' is not defined` (o `np`, `stats`…) | No habéis ejecutado la celda de las librerías, o Colab se reinició | Ejecutad las celdas **desde el paso 0** hasta la que falla, en orden |
| La comprobación da ❌ aunque creéis que está bien | Habéis corregido el código pero no habéis vuelto a ejecutar la celda del ejercicio | Ejecutad la celda del ejercicio **otra vez** y después la de comprobación |
| Sale un error al descargar el fichero (`URLError`, `FileNotFoundError`) | Sin internet o sin acceso a GitHub | Probad otra vez; si sigue, avisad al docente (puede dar los ficheros `ocupacion_diaria.csv` y `reservas_turisdata.csv`) |
| Seguís sin verlo | A veces pasa | 1) Preguntad a vuestra pareja; 2) después, al docente; 3) **cuando hayáis terminado o agotado las pistas**, ejecutad la «Solución de referencia» del ejercicio. Primero intentadlo: lo que copiéis, tenéis que saber explicarlo |

## 🆘 Soluciones de referencia

- Ejecutad estas celdas **solo después de intentarlo**.
- Cada ejercicio tiene la suya.
- Si lo dejáis para después, ejecutadlas para poder seguir con el paso siguiente.

In [ ]:
#@title 🆘 Solución del ejercicio 1 · Describir (ábrela solo después de intentarlo)
resumen = {
    "media": round(serie.mean(), 2),
    "mediana": round(serie.median(), 2),
    "desviacion": round(serie.std(), 2),
    "iqr": round(serie.quantile(0.75) - serie.quantile(0.25), 2),
}
print("Solución del ejercicio 1 aplicada: ahora ejecutad la celda 2.3")

In [ ]:
#@title 🆘 Solución del ejercicio 5 · Intervalo de confianza (ábrela solo después de intentarlo)
p_hat = reservas["cancelada"].mean()
n_res = len(reservas)
error = np.sqrt(p_hat * (1 - p_hat) / n_res)
ic_tasa = (round(p_hat - 1.96 * error, 4), round(p_hat + 1.96 * error, 4))
print("Solución del ejercicio 5 aplicada: ahora ejecutad la celda 4.2")

## Para reflexionar (con vuestra pareja)

**Parte A · Describir**

1. En 2.1 la media era 76 y la mediana 70. Si el último dato fuera `1000` en vez de `100`, ¿cuál de las dos cambiaría más? *(probadlo)*
2. ¿Qué os dice que la desviación sea 9.61? ¿Es mucha o poca variación en una ocupación que va del 47 % al 97 %?

**Parte B · Intervalo de confianza**

1. Si en vez de 6000 reservas tuviéramos 600, ¿el intervalo sería más ancho o más estrecho? ¿Por qué?
2. El intervalo va de 44,6 % a 47,2 %. ¿Sería razonable que la tasa real fuera 50 %? ¿Y 46 %?

## Qué habéis aprendido

**Parte A · Describir**

- Una serie larga se resume con el **centro** (media y mediana) y la **dispersión** (desviación e IQR).
- La media se deja arrastrar por los valores extremos; la mediana no.
- Los datos reales se resumen con `serie.mean()`, `serie.median()`, `serie.std()` y `serie.quantile(…)`.

**Parte B · Intervalo de confianza**

- Una cifra calculada con una muestra necesita un **margen de error**: el **intervalo de confianza**.
- Para una proporción: `p ± 1.96 × √(p (1 − p) / n)`.
- Con más datos, el intervalo es más estrecho: la cifra es más fiable.